# Appendix: speed, measured

> The measurements behind notebook 101: what Kokoro's own speed loses, and three naive and six real ways to speed up
> audio, judged by a speech recogniser and a quality predictor.

Notebook 101 says Kokoro's own `speed` drops sounds and that Sonic is the best way to speed up audio. This is where
those claims come from. It takes about twenty minutes, so `nbdev-test` skips it.

In [ ]:
import difflib
import re

import librosa
import numpy as np
import pandas as pd
import pylibrb
import pytsmod
import torch
from audiotsm import wsola
from audiotsm.io.array import ArrayReader, ArrayWriter
from fastcore.test import test_eq
from IPython.display import Audio, display
from transformers import pipeline as hf_pipeline

from text_to_speech.data import metadata
from text_to_speech.sonic import sonic
from text_to_speech.voice import FRAME, SR, analyse, model, phonemize, pipeline

/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
W1101 22:42:24.841000 74500 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W1101 22:42:24.879000 74500 .venv/lib/python3.13/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1101 22:42:24.928000 74500 .venv/lib/python3.13/site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclas

## Kokoro's own speed

Kokoro rounds each phoneme's duration to a whole number of frames, 600 samples (25 ms) each, and never fewer than one.
`analyse` keeps those frame counts in `frames`. `<` and `>` are the silences before and after the speech, and stress marks get
frames of their own.

In [ ]:
text = "Speed comes from the voice itself, not from stretching the audio afterwards."
u1, u2 = analyse(text), analyse(text, speed=2)
frames = pd.DataFrame(
    {"phoneme": ["<", *[p for p in u1.phonemes if p in model.vocab], ">"], "1x": u1.frames, "2x": u2.frames}
)
print(f"{frames['1x'].mean():.1f} frames per phoneme at 1x, and {(frames['2x'] == 1).mean():.0%} get just one at 2x")
frames.head(32).T

2.5 frames per phoneme at 1x, and 86% get just one at 2x


/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
phoneme,<,s,p,ˈ,i,d,,k,ˈ,ʌ,m,z,,f,ɹ,ʌ,m,,ð,ə,,v,ˈ,Y,s,,ɪ,t,s,ˈ,ɛ,l
1x,14,3,2,3,3,2,2,2,2,2,2,2,1,1,1,1,1,1,1,1,2,2,3,3,2,2,2,3,3,3,3,3
2x,7,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,1,1,1,1,2,1,2,1,2


At 1x a phoneme gets two or three frames. At 2x nearly every phoneme gets exactly one, and the ones that should get
one and a half are rounded *down*: the /s/ of "speed", the vowel of "voice", the /ʧ/ of "stretching". A single 25 ms
frame is one feature vector, and the decoder's convolutions blur it into its neighbours, so short consonants vanish.
It's also why 3x is barely shorter than 2x: almost everything is already at the floor. And the silence before the
speech halves too, which takes the start of the first word with it.

### Counting what's lost

Ears are the judge, but to compare fixes we want a number. So we transcribe with Whisper and count the fraction of
words it hears, in order. Whisper also struggles with an abrupt start, so we pad the audio with silence first.

To try other retimings we need the one line of Kokoro that applies `speed`. `render` is notebook 100's `analyse` and
`decode` with that line, `round(duration / speed).clamp(min=1)`, passed in as `retime`.

In [ ]:
@torch.no_grad()
def render(phonemes, retime, voice="af_heart"):
    "Kokoro's `forward_with_tokens`, with the rounding of durations to frames done by `retime`."
    ref_s = pipeline(voice).load_voice(voice)[len(phonemes) - 1]
    ids = torch.LongTensor([[0, *[model.vocab[p] for p in phonemes if p in model.vocab], 0]])
    n = ids.shape[1]
    lengths, mask = torch.tensor([n]), torch.zeros(1, n, dtype=torch.bool)
    s = ref_s[:, 128:]
    bert = model.bert_encoder(model.bert(ids, attention_mask=(~mask).int())).transpose(-1, -2)
    d = model.predictor.text_encoder(bert, s, lengths, mask)
    duration = torch.sigmoid(model.predictor.duration_proj(model.predictor.lstm(d)[0])).sum(-1).squeeze()  # In frames
    frames = retime(duration)
    aln = torch.zeros(1, n, int(frames.sum()))  # Which phoneme each frame belongs to
    aln[0, torch.repeat_interleave(torch.arange(n), frames), torch.arange(int(frames.sum()))] = 1
    F0, N = model.predictor.F0Ntrain(d.transpose(-1, -2) @ aln, s)
    return model.decoder(model.text_encoder(ids, lengths, mask) @ aln, F0, N, ref_s[:, :128]).squeeze().numpy()


def kokoro_speed(speed):
    return lambda duration: torch.round(duration / speed).clamp(min=1).long()


def keep_lead_in(speed):
    "Kokoro's speed, except the silences before and after the speech keep their 1x length."

    def retime(duration):
        frames = kokoro_speed(speed)(duration)
        frames[[0, -1]] = torch.round(duration[[0, -1]]).long()
        return frames

    return retime

In [ ]:
test_eq(len(render(u2.phonemes, kokoro_speed(2))), int(u2.frames.sum()) * FRAME)

In [ ]:
whisper = hf_pipeline("automatic-speech-recognition", model="openai/whisper-small.en")


def heard(audio):
    "What Whisper hears, with half a second of silence either side."
    audio = librosa.resample(np.pad(audio, SR // 2), orig_sr=SR, target_sr=16_000)
    return whisper({"raw": audio, "sampling_rate": 16_000})["text"]


def words(t):
    return re.sub(r"[^a-z' ]", " ", t.lower()).split()


def recall(text, audio):
    "Fraction of the words in `text` that Whisper hears, in order."
    blocks = difflib.SequenceMatcher(a=words(text), b=words(heard(audio)), autojunk=False).get_matching_blocks()
    return sum(b.size for b in blocks) / len(words(text))

Loading weights: 100%|##########| 479/479 [00:00<00:00, 3440.63it/s]


Four ways to go faster, on our sentence and nine held-out LJSpeech transcripts. Two of them time-stretch, which keeps
pitch and squeezes the waveform after rendering: on its own, and on top of Kokoro at 1.3x, where few phonemes are
rounded down yet. Whisper can't follow even clean 3x speech, so we stop at 2x.

In [ ]:
texts = [text, *[t for t in metadata().query("id.str.startswith('LJ050')").normalized_text if len(t) < 200][:9]]
phonemes = [phonemize(t)[0] for t in texts]
normal = [render(ps, kokoro_speed(1)) for ps in phonemes]


def stretched(speed, model_speed=1):
    "Kokoro at `model_speed`, then time-stretched to exactly `1 / speed` of the 1x length."
    out = []
    for ps, slow in zip(phonemes, normal):
        fast = render(ps, keep_lead_in(model_speed))
        out.append(librosa.effects.time_stretch(fast, rate=len(fast) * speed / len(slow)))
    return out


rows = [(1, "Kokoro", 1, np.mean([recall(t, a) for t, a in zip(texts, normal)]))]
for speed in [1.5, 2]:
    methods = {
        "Kokoro's speed": [render(ps, kokoro_speed(speed)) for ps in phonemes],
        "Kokoro's speed, keeping the lead-in": [render(ps, keep_lead_in(speed)) for ps in phonemes],
        "time-stretch": stretched(speed),
        "Kokoro at 1.3x, then time-stretch": stretched(speed, 1.3),
    }
    for name, audios in methods.items():
        actual = sum(map(len, normal)) / sum(map(len, audios))
        rows.append((speed, name, actual, np.mean([recall(t, a) for t, a in zip(texts, audios)])))
    if speed == 2:
        for name, audios in methods.items():
            print(name)
            display(Audio(audios[0], rate=SR))
pd.DataFrame(rows, columns=["speed", "method", "actual speed", "word recall"]).round(2)

Kokoro's speed
Kokoro's speed, keeping the lead-in
time-stretch
Kokoro at 1.3x, then time-stretch


/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/librosa/effects.py:448: FutureWarning: The `hop_length` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/librosa/effects.py:448: FutureWarning: The `n_fft` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/numba/np/ufunc/dufunc.py:303: RuntimeWarning: invalid value encountered in cast
  return super().__call__(*args, **kws)


,speed,method,actual speed,word recall
0,1.0,Kokoro,1.00,0.98
1,1.5,Kokoro's speed,1.56,0.88
2,1.5,"Kokoro's speed, keeping the lead-in",1.51,0.94
3,1.5,time-stretch,1.50,0.94
4,1.5,"Kokoro at 1.3x, then time-stretch",1.50,0.95
5,2.0,Kokoro's speed,1.84,0.75
6,2.0,"Kokoro's speed, keeping the lead-in",1.73,0.89
7,2.0,time-stretch,2.00,0.91
8,2.0,"Kokoro at 1.3x, then time-stretch",2.00,0.89


Keeping the lead-in is the one clear win: it brings the first word back and fixes most of the damage at 1.5x, for
free. What's left is the frame grid, and at 2x it still loses a word in eight. Past that, nothing here beats a plain
time-stretch by more than a word across these ten sentences, which is noise. But the time-stretch here is librosa's
phase vocoder, and the next section finds it is the worst of the real stretchers: it smears, which is what audio-reader
heard. With Sonic instead, Whisper hears every word at 2x that it hears at 1x.

Two other retimings did worse, so they aren't here. Fractional durations, sharing a frame between neighbouring
phonemes, sound worse than rounding (a third of words lost at 2x): the decoder never saw blended frames in training.
Protecting short phonemes by squeezing long ones harder never gets past about 1.4x, and loses more words on the way.

The lesson for the from-scratch line is about frame rate. At 1x this voice says about 16 phonemes a second, so 3x is
about 50. A model that must speak at 3x needs comfortably more than 50 frames a second. Kokoro has 40. Our mel
spectrogram, at a hop of 256 samples, has 86.

## Nine ways to speed up audio

First, the naive recipe: render at 1x, play it faster, and correct the pitch.

- **Resample only.** Playing faster shortens the audio and raises the pitch and the formants by the same factor: the
  chipmunk. Not a candidate, just the starting point.
- **Resample, then pitch-shift down.** The literal recipe. But a pitch shift that keeps the length is itself a
  time-stretch plus a resample (librosa's uses the phase vocoder), so this is time-stretching in disguise.
- **Naive overlap-add.** Chop the audio into fixed 43 ms windows, read them further apart than they're written, and
  add them up. No alignment of any kind, so neighbouring windows meet out of phase.

Then six real time-stretchers:

- **Phase vocoder** (librosa). Take an STFT, space the frames closer together, and fix up each frequency's phase.
  Phases drift apart between frequencies, which gives speech a phasey, reverberant smear and softens consonants.
- **WSOLA** (audiotsm). Cut the waveform into overlapping windows and drop some, choosing each cut point where the
  waveform lines up with the previous window so nothing clicks.
- **HPTSM** (pytsmod). Split the sound into harmonic and percussive parts, phase-vocode the harmonics and
  overlap-add the clicks and hisses, then mix them back.
- **TD-PSOLA** (pytsmod). Track the pitch, centre a window on every glottal pulse, and drop whole pitch periods.
  The classic speech method.
- **Rubber Band**, R2 "faster" and R3 "finer" engines (pylibrb). A phase vocoder with transient detection and phase
  locking, and the industry standard. GPL, or a commercial licence.
- **Sonic**. Also drops whole pitch periods, but finds the period itself rather than needing a pitch tracker. It was
  written for blind listeners who run screen readers at several times normal speed, and Android's playback speed
  control uses a port of it. Apache-licensed, one C file, vendored in `vendor/sonic`.

In [ ]:
def resample_only(x, speed):
    "Play it faster: shorter, and higher by the same factor."
    return librosa.resample(x, orig_sr=SR * speed, target_sr=SR)


def resample_then_pitch_down(x, speed):
    "Speed up by resampling, then pitch-shift back down."
    return librosa.effects.pitch_shift(resample_only(x, speed), sr=SR, n_steps=-12 * np.log2(speed))


def naive_ola(x, speed, win=1024):
    "Fixed Hann windows, read `speed` times further apart than they are written. No alignment."
    hop_out = win // 2
    hop_in = round(hop_out * speed)
    w = np.hanning(win)
    n = (len(x) - win) // hop_in + 1
    out = np.zeros(n * hop_out + win, dtype=np.float32)
    for i in range(n):
        out[i * hop_out : i * hop_out + win] += x[i * hop_in : i * hop_in + win] * w
    return out


def phase_vocoder(x, speed):
    return librosa.effects.time_stretch(x, rate=speed)


def wsola_audiotsm(x, speed):
    out = ArrayWriter(1)
    wsola(1, speed=speed).run(ArrayReader(x[None].astype(np.float32)), out)
    return out.data[0]


def hptsm(x, speed):
    return pytsmod.hptsm(x, 1 / speed)


def tdpsola(x, speed, hop=120):
    "TD-PSOLA needs a pitch track: pYIN every 5 ms, 0 where unvoiced."
    f0, _, _ = librosa.pyin(x, fmin=60, fmax=500, sr=SR, frame_length=1024, hop_length=hop)
    return pytsmod.tdpsola(x, SR, np.nan_to_num(f0), alpha=1 / speed, p_hop_size=hop, p_win_size=1024)


def rubberband(engine):
    def stretch(x, speed):
        options = pylibrb.Option.PROCESS_OFFLINE | engine | pylibrb.Option.FORMANT_PRESERVED
        stretcher = pylibrb.RubberBandStretcher(
            sample_rate=SR, channels=1, options=options, initial_time_ratio=1 / speed
        )
        x = x[None].astype(np.float32)
        stretcher.study(x, final=True)
        stretcher.process(x, final=True)
        return stretcher.retrieve_available()[0]

    return stretch


stretchers = {
    "resample only": resample_only,
    "resample, then pitch down": resample_then_pitch_down,
    "naive overlap-add": naive_ola,
    "phase vocoder": phase_vocoder,
    "WSOLA": wsola_audiotsm,
    "HPTSM": hptsm,
    "TD-PSOLA": tdpsola,
    "Rubber Band R2": rubberband(pylibrb.Option.ENGINE_FASTER),
    "Rubber Band R3": rubberband(pylibrb.Option.ENGINE_FINER),
    "Sonic": sonic,
}

The same ten sentences as above, rendered once at 1x and stretched by each method. Kokoro's own speed, keeping
the lead-in silence, is the reference.

In [ ]:
utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).eval()


def mos(audio):
    with torch.no_grad():
        return utmos(torch.tensor(audio, dtype=torch.float32)[None], SR).item()


text = "Speed comes from the voice itself, not from stretching the audio afterwards."
texts = [text, *[t for t in metadata().query("id.str.startswith('LJ050')").normalized_text if len(t) < 200][:9]]
phonemes = [phonemize(t)[0] for t in texts]
normal = [render(ps, keep_lead_in(1)) for ps in phonemes]


def score(name, speed, audios):
    return {
        "method": name,
        "speed": speed,
        "actual": sum(map(len, normal)) / sum(map(len, audios)),
        "recall": np.mean([recall(t, a) for t, a in zip(texts, audios)]),
        "mos": np.mean([mos(a) for a in audios]),
    }


rows, examples = [score("Kokoro", 1, normal)], {}
for speed in [1.5, 2, 2.5, 3]:
    methods = {"Kokoro's speed": [render(ps, keep_lead_in(speed)) for ps in phonemes]}
    methods |= {name: [stretch(a, speed) for a in normal] for name, stretch in stretchers.items()}
    rows += [score(name, speed, audios) for name, audios in methods.items()]
    examples[speed] = {name: audios[0] for name, audios in methods.items()}
results = pd.DataFrame(rows)
results.pivot(index="method", columns="speed", values=["recall", "mos"]).round(2)

Using cache found in /Users/henrydashwood/.cache/torch/hub/tarepan_SpeechMOS_v1.2.0
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/librosa/effects.py:448: FutureWarning: The `hop_length` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/librosa/effects.py:448: FutureWarning: The `n_fft` parameter is deprecated as of 1.0 and will be removed in 1.1. It is unused in the current implementation.
  stft_stretch = core.phase_vocoder(
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/librosa/effects.py:448: FutureWarning: The `hop_l

recall                    ...   mos                  
speed                        1.0   1.5   2.0   2.5  ...   1.5   2.0   2.5   3.0
method                                              ...                        
HPTSM                        NaN  0.98  0.98  0.95  ...  3.49  1.72  1.33  1.31
Kokoro                      0.98   NaN   NaN   NaN  ...   NaN   NaN   NaN   NaN
Kokoro's speed               NaN  0.93  0.88  0.77  ...  4.37  4.22  4.00  3.96
Rubber Band R2               NaN  0.98  0.98  0.96  ...  2.57  1.39  1.30  1.28
Rubber Band R3               NaN  0.98  0.98  0.93  ...  3.66  1.98  1.34  1.31
Sonic                        NaN  0.98  0.98  0.95  ...  4.28  2.93  1.39  1.33
TD-PSOLA                     NaN  0.98  0.98  0.94  ...  3.71  1.57  1.34  1.31
WSOLA                        NaN  0.98  0.93  0.71  ...  4.06  2.61  1.33  1.31
naive overlap-add            NaN  0.98  0.97  0.95  ...  3.46  2.09  1.36  1.32
phase vocoder                NaN  0.95  0.86  0.55  ...  1.32  1.31  1.31  1.30
resample only                NaN  0.98  0.73  0.08  ...  1.98  1.27  1.27  1.29
resample, then pitch down    NaN  0.96  0.27  0.06  ...  1.30  1.30  1.29  1.30

[12 rows x 10 columns]

Speeds actually reached, where they differ from what was asked:

In [ ]:
results[(results.actual - results.speed).abs() > 0.03].round(2)

,method,speed,actual,recall,mos
12,Kokoro's speed,2.0,1.73,0.88,4.22
23,Kokoro's speed,2.5,1.94,0.77,4.00
26,naive overlap-add,2.5,2.46,0.95,1.36
28,WSOLA,2.5,3.05,0.71,1.33
34,Kokoro's speed,3.0,2.03,0.74,3.96
37,naive overlap-add,3.0,2.95,0.79,1.32
39,WSOLA,3.0,4.06,0.17,1.31


Our sentence at 2x, three ways. Then Sonic at every speed.

In [ ]:
for name in ["phase vocoder", "Kokoro's speed", "Sonic"]:
    print(name)
    display(Audio(examples[2][name], rate=SR))
for speed in [1.5, 2, 2.5, 3]:
    print(f"Sonic {speed}x")
    display(Audio(examples[speed]["Sonic"], rate=SR))

phase vocoder
Kokoro's speed
Sonic
Sonic 1.5x
Sonic 2x
Sonic 2.5x
Sonic 3x


## Results

From the run on 1 November 2026 (reruns move by a point or two), word recall (Whisper) and predicted MOS (UTMOS) over the ten sentences. At 1x Kokoro scores
97.6% and 4.52.

| method | 1.5x recall | 1.5x MOS | 2x recall | 2x MOS | 2.5x recall | 3x recall |
|---|---|---|---|---|---|---|
| Kokoro's speed (reaches 1.51x, 1.73x, 1.94x, 2.03x) | 94.1% | **4.36** | 87.3% | **4.22** | 73.9% | 73.3% |
| resample only | 97.6% | 1.98 | 72.7% | 1.27 | 10.7% | 0.8% |
| resample, then pitch down | 90.7% | 1.30 | 33.8% | 1.30 | 7.6% | 7.2% |
| naive overlap-add | 97.6% | 3.47 | 97.1% | 2.06 | **94.0%** | 78.4% |
| phase vocoder | 95.1% | 1.32 | 86.0% | 1.31 | 52.8% | 15.8% |
| WSOLA (overshoots: 3.05x, 4.06x) | 97.6% | 4.03 | 92.8% | 2.62 | 62.6% | 14.6% |
| HPTSM | 97.6% | 3.52 | 97.6% | 1.71 | **94.6%** | 77.7% |
| TD-PSOLA | 97.6% | 3.76 | 97.6% | 1.56 | 93.7% | 82.5% |
| Rubber Band R2 | 97.6% | 2.51 | 95.1% | 1.45 | 92.1% | 74.4% |
| Rubber Band R3 | 97.6% | 3.69 | 97.6% | 2.05 | 93.1% | **89.9%** |
| **Sonic** | **97.6%** | 4.29 | **97.6%** | 2.92 | 93.7% | 88.8% |

- **The naive recipe is the worst of all.** Resampling then pitch-shifting back loses two thirds of the words at
  2x, worse than not correcting the pitch at all. librosa's pitch shift is a phase-vocoder stretch plus a resample, so
  it smears, and the resampling moves the formants too.
- **Naive overlap-add is surprisingly good**: as intelligible as Sonic up to 2.5x. But it sounds rougher (2.06 against
  2.92 at 2x), because its windows meet out of phase, and by 3x it loses a fifth of the words to Sonic's tenth. Lining
  the cuts up on pitch periods is exactly what Sonic adds, and that's what the gap measures.
- **The phase vocoder is the worst real stretcher**, with a MOS of 1.3 already at 1.5x. That's the smear audio-reader
  heard, and it's what notebook 100 compared against. Time-stretching wasn't the problem; this time-stretcher was.
- **Sonic is the best overall.** At 2x, Whisper hears every word it heard at 1x, and UTMOS prefers it to every other
  stretcher. At 3x it is about a point behind the best. It also costs almost nothing: all ten sentences, at any
  speed, in under a twentieth of a second.
- **Kokoro's own speed sounds the most natural to UTMOS**, but it never gets past 2x, and it loses words the
  stretchers keep.
- **UTMOS hits its floor**, about 1.3, for every stretcher from 2.5x. It judges fast speech as bad speech, whatever
  the method, so above 2x only recall and your ears can tell methods apart.
- audiotsm's WSOLA overshoots the speed it is asked for from 2.5x, so its numbers there aren't comparable.

So the player uses Sonic for speed. It suits the phones too: Android's own speed control already uses a port of it,
and on iOS it compiles as-is. Rubber Band, the only other contender at 3x, would need a commercial licence for the
App Store.

Whisper and UTMOS are not ears. Listen to the examples above before believing the table, especially at 3x.

## Things to try

- Kokoro's own speed up to 1.3x, where it rounds few phonemes down, then Sonic for the rest. Does UTMOS prefer it?
- Sonic has a higher-quality mode (`sonicSetQuality` on its streaming interface). Is it audibly better at 3x?
- audio-reader's listeners chose speeds by ear. Find the speed where each method stops being comfortable to listen to
  for a whole paragraph.